# YOLOX Confidence Boost Fine-Tuning

**Goal:** Fix the confidence ceiling issue (~53% max) while preserving 100% classification accuracy.

## Problem Analysis
The 100-epoch model shows:
- 100% classification accuracy (every detected label is correct)
- Good recall (~70-80%)
- Confidence ceiling at 46-53%

This suggests the model learned correct classifications but didn't learn strong objectness scores. The sigmoid(~0.1) ≈ 52% pattern indicates the objectness logits are too close to zero.

## Combined Approach
This notebook implements **ALL** recommended fixes:

1. **Extended No-Aug Phase** (40 epochs with no augmentation)
2. **Lower Learning Rate** (10-20x lower than initial training)
3. **Increased Objectness Loss Weight** (2-3x higher obj_loss weight)
4. **More Real Images** (increase real:synthetic ratio)

## Prerequisites
1. 100-epoch checkpoint at: `/My Drive/hazmat_models/100epoch_realistic/100epoch_realistic_ckpt.pth`
2. Real images dataset at: `/My Drive/HazProML/data/real_images_finetune/`
3. Synthetic dataset at: `/My Drive/synthetic_realistic/`
4. Runtime set to **GPU T4** (Runtime > Change runtime type > T4 GPU)

## Cell 1: Mount Google Drive & Verify GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU found! Please enable GPU in Runtime > Change runtime type")

Mounted at /content/drive
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.7 GB


## Cell 2: Configuration

### Key Changes for Confidence Boost:
- `OBJ_LOSS_WEIGHT = 2.0` (default is 1.0) - Forces model to learn stronger objectness
- `BASIC_LR = 0.0005 / 64` (vs 0.01 / 64) - 20x lower LR preserves learned features
- `NO_AUG_EPOCHS = 40` (vs 15) - Extended clean training for confidence calibration
- `REAL_IMAGE_RATIO = 0.4` - 40% real images (vs ~12% before)

In [2]:
import os
import json

# ============================================
# PATHS - UPDATE THESE FOR YOUR SETUP
# ============================================

DRIVE_ROOT = "/content/drive/MyDrive"

# Pre-trained checkpoint (100 epochs)
PRETRAINED_CKPT = f"{DRIVE_ROOT}/hazmat_models/100epoch_realistic/100epoch_realistic_ckpt.pth"

# Alternative paths to try if the above doesn't exist:
ALTERNATIVE_CKPTS = [
    f"{DRIVE_ROOT}/hazmat_models/100epoch_realistic/latest_ckpt.pth",
    f"{DRIVE_ROOT}/hazmat_models/100epoch_realistic/epoch_100_ckpt.pth",
    f"{DRIVE_ROOT}/HazProML/models/mixed_background_100epoch/checkpoint_epoch_100.pth",
]

# Real images dataset
REAL_IMAGES_DIR = f"{DRIVE_ROOT}/HazProML/data/real_images_finetune"

# Synthetic dataset (for mixing)
SYNTHETIC_DIR = f"{DRIVE_ROOT}/synthetic_realistic"

# Class mapping
CLASS_MAPPING = f"{DRIVE_ROOT}/class_mapping.json"

# Output directory
DRIVE_OUTPUT = f"{DRIVE_ROOT}/hazmat_models/confidence_boost"

# ============================================
# TRAINING PARAMETERS - CONFIDENCE BOOST
# ============================================

# Total fine-tuning epochs
MAX_EPOCHS = 60

# Extended no-aug phase (40 epochs of clean training)
# This helps the model learn precise confidence scores
NO_AUG_EPOCHS = 40

# Learning rate: 20x lower than initial training
# Initial was 0.01/64, we use 0.0005/64
BASIC_LR = 0.0005 / 64.0

# Objectness loss weight: 2x higher than default
# This forces the model to produce stronger objectness scores
OBJ_LOSS_WEIGHT = 2.0  # Default is 1.0

# Alternative: Even higher objectness weight (3x)
# OBJ_LOSS_WEIGHT = 3.0

# Batch size
BATCH_SIZE = 16  # Reduce to 8 if OOM

# Input size (must match pre-trained model)
INPUT_SIZE = (640, 640)

# Real image ratio in training set
# 0.4 = 40% real, 60% synthetic
REAL_IMAGE_RATIO = 0.4

# Checkpoint backup interval
BACKUP_INTERVAL = 10

# ============================================
# LOCAL WORKING DIRECTORY
# ============================================
LOCAL_DATA_DIR = '/content/data/confidence_boost'

# ============================================
# VERIFY PATHS
# ============================================
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Find valid checkpoint
ckpt_path = None
if os.path.exists(PRETRAINED_CKPT):
    ckpt_path = PRETRAINED_CKPT
else:
    for alt in ALTERNATIVE_CKPTS:
        if os.path.exists(alt):
            ckpt_path = alt
            break

print("="*60)
print("CONFIDENCE BOOST FINE-TUNING CONFIGURATION")
print("="*60)
print(f"\nCheckpoint: {ckpt_path}")
if ckpt_path:
    print(f"  Status: Found ({os.path.getsize(ckpt_path)/1024/1024:.1f} MB)")
else:
    print(f"  Status: NOT FOUND")
    print(f"  Tried: {PRETRAINED_CKPT}")
    for alt in ALTERNATIVE_CKPTS:
        print(f"  Tried: {alt}")

print(f"\nReal images: {REAL_IMAGES_DIR}")
print(f"  Status: {'Found' if os.path.exists(REAL_IMAGES_DIR) else 'NOT FOUND'}")

print(f"\nSynthetic data: {SYNTHETIC_DIR}")
print(f"  Status: {'Found' if os.path.exists(SYNTHETIC_DIR) else 'NOT FOUND'}")

print(f"\nClass mapping: {CLASS_MAPPING}")
print(f"  Status: {'Found' if os.path.exists(CLASS_MAPPING) else 'NOT FOUND'}")

print(f"\n" + "-"*60)
print("KEY CONFIDENCE BOOST SETTINGS:")
print("-"*60)
print(f"  Objectness Loss Weight: {OBJ_LOSS_WEIGHT}x (default: 1.0)")
print(f"  Learning Rate: {BASIC_LR * 64:.6f} (20x lower than initial)")
print(f"  No-Aug Epochs: {NO_AUG_EPOCHS} out of {MAX_EPOCHS} total")
print(f"  Real Image Ratio: {REAL_IMAGE_RATIO*100:.0f}%")
print(f"\nOutput: {DRIVE_OUTPUT}")
print("="*60)

# Load class count
if os.path.exists(CLASS_MAPPING):
    with open(CLASS_MAPPING, 'r') as f:
        class_map = json.load(f)
    NUM_CLASSES = len(class_map)
    print(f"\nClasses: {NUM_CLASSES}")
else:
    NUM_CLASSES = 92  # Default
    print(f"\nUsing default class count: {NUM_CLASSES}")

# Final validation
if not ckpt_path:
    raise FileNotFoundError("No checkpoint found! Please update PRETRAINED_CKPT path.")

CONFIDENCE BOOST FINE-TUNING CONFIGURATION

Checkpoint: /content/drive/MyDrive/HazProML/models/mixed_background_100epoch/checkpoint_epoch_100.pth
  Status: Found (38.9 MB)

Real images: /content/drive/MyDrive/HazProML/data/real_images_finetune
  Status: Found

Synthetic data: /content/drive/MyDrive/synthetic_realistic
  Status: Found

Class mapping: /content/drive/MyDrive/class_mapping.json
  Status: Found

------------------------------------------------------------
KEY CONFIDENCE BOOST SETTINGS:
------------------------------------------------------------
  Objectness Loss Weight: 2.0x (default: 1.0)
  Learning Rate: 0.000500 (20x lower than initial)
  No-Aug Epochs: 40 out of 60 total
  Real Image Ratio: 40%

Output: /content/drive/MyDrive/hazmat_models/confidence_boost

Classes: 92


## Cell 3: Install Dependencies

In [ ]:
%%capture
# Install required packages (suppress output for cleaner logs)
!pip install cython
!pip install pycocotools
!pip install thop
!pip install loguru
!pip install tabulate

# Clone YOLOX repository
import os
if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

# Install YOLOX
%cd /content/YOLOX
!pip install -v -e .

In [ ]:
# Verify installation
import sys
sys.path.insert(0, '/content/YOLOX')

try:
    from yolox.exp import get_exp
    from yolox.utils import get_model_info
    print("YOLOX installed successfully!")
except ImportError as e:
    print(f"YOLOX installation failed: {e}")
    raise

## Cell 4: Prepare Combined Dataset

Creates a dataset with the specified real:synthetic ratio.

**Key:** More real images help the model learn realistic objectness patterns.

In [ ]:
import os
import shutil
import random
from tqdm import tqdm
from pathlib import Path

# Clean and create directories
!rm -rf /content/data
os.makedirs(f'{LOCAL_DATA_DIR}/images/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/images/val', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/val', exist_ok=True)

# ============================================
# STEP 1: Count available images
# ============================================
real_train_imgs = f"{REAL_IMAGES_DIR}/images/train"
real_train_labels = f"{REAL_IMAGES_DIR}/labels/train"
syn_train_imgs = f"{SYNTHETIC_DIR}/images/train"
syn_train_labels = f"{SYNTHETIC_DIR}/labels/train"

# Count real images
real_count = 0
if os.path.exists(real_train_imgs):
    real_files = [f for f in os.listdir(real_train_imgs) if f.endswith(('.jpg', '.png'))]
    real_count = len(real_files)

# Count synthetic images
syn_count = 0
if os.path.exists(syn_train_imgs):
    syn_files = [f for f in os.listdir(syn_train_imgs) if f.endswith(('.jpg', '.png'))]
    syn_count = len(syn_files)

print(f"Available images:")
print(f"  Real: {real_count}")
print(f"  Synthetic: {syn_count}")

# ============================================
# STEP 2: Calculate mix ratio
# ============================================
# Target: REAL_IMAGE_RATIO of training data should be real
# If we have N real images, we need (1-REAL_IMAGE_RATIO)/REAL_IMAGE_RATIO * N synthetic

if real_count > 0:
    # Calculate how many synthetic to use based on real count and ratio
    target_synthetic = int(real_count * (1 - REAL_IMAGE_RATIO) / REAL_IMAGE_RATIO)
    synthetic_to_use = min(target_synthetic, syn_count)

    actual_ratio = real_count / (real_count + synthetic_to_use)
    print(f"\nTarget ratio: {REAL_IMAGE_RATIO*100:.0f}% real")
    print(f"Actual ratio: {actual_ratio*100:.1f}% real")
    print(f"Using {real_count} real + {synthetic_to_use} synthetic = {real_count + synthetic_to_use} total")
else:
    # No real images, use all synthetic
    synthetic_to_use = syn_count
    print(f"\nNo real images found, using {synthetic_to_use} synthetic only")

# ============================================
# STEP 3: Copy real images
# ============================================
if real_count > 0:
    print(f"\nCopying {real_count} real images...")
    for f in tqdm(real_files, desc="Real images"):
        src_img = f"{real_train_imgs}/{f}"
        dst_img = f"{LOCAL_DATA_DIR}/images/train/real_{f}"
        shutil.copy2(src_img, dst_img)

        # Copy label
        label_name = os.path.splitext(f)[0] + '.txt'
        src_label = f"{real_train_labels}/{label_name}"
        if os.path.exists(src_label):
            dst_label = f"{LOCAL_DATA_DIR}/labels/train/real_{label_name}"
            shutil.copy2(src_label, dst_label)

# ============================================
# STEP 4: Copy synthetic images (shuffled subset)
# ============================================
if synthetic_to_use > 0:
    print(f"\nCopying {synthetic_to_use} synthetic images...")
    random.shuffle(syn_files)
    selected_syn = syn_files[:synthetic_to_use]

    for f in tqdm(selected_syn, desc="Synthetic images"):
        src_img = f"{syn_train_imgs}/{f}"
        dst_img = f"{LOCAL_DATA_DIR}/images/train/syn_{f}"
        shutil.copy2(src_img, dst_img)

        # Copy label
        label_name = os.path.splitext(f)[0] + '.txt'
        src_label = f"{syn_train_labels}/{label_name}"
        if os.path.exists(src_label):
            dst_label = f"{LOCAL_DATA_DIR}/labels/train/syn_{label_name}"
            shutil.copy2(src_label, dst_label)

# ============================================
# STEP 5: Copy validation images (real only if available)
# ============================================
print("\nCopying validation images...")
real_val_imgs = f"{REAL_IMAGES_DIR}/images/val"
real_val_labels = f"{REAL_IMAGES_DIR}/labels/val"
syn_val_imgs = f"{SYNTHETIC_DIR}/images/val"
syn_val_labels = f"{SYNTHETIC_DIR}/labels/val"

val_count = 0

# Prefer real validation images
if os.path.exists(real_val_imgs):
    for f in os.listdir(real_val_imgs):
        if f.endswith(('.jpg', '.png')):
            shutil.copy2(f"{real_val_imgs}/{f}", f"{LOCAL_DATA_DIR}/images/val/{f}")
            label_name = os.path.splitext(f)[0] + '.txt'
            if os.path.exists(f"{real_val_labels}/{label_name}"):
                shutil.copy2(f"{real_val_labels}/{label_name}", f"{LOCAL_DATA_DIR}/labels/val/{label_name}")
            val_count += 1

# Add synthetic validation if not enough real
if val_count < 50 and os.path.exists(syn_val_imgs):
    syn_val_files = [f for f in os.listdir(syn_val_imgs) if f.endswith(('.jpg', '.png'))]
    for f in syn_val_files[:100 - val_count]:
        shutil.copy2(f"{syn_val_imgs}/{f}", f"{LOCAL_DATA_DIR}/images/val/syn_{f}")
        label_name = os.path.splitext(f)[0] + '.txt'
        if os.path.exists(f"{syn_val_labels}/{label_name}"):
            shutil.copy2(f"{syn_val_labels}/{label_name}", f"{LOCAL_DATA_DIR}/labels/val/syn_{label_name}")
        val_count += 1

# ============================================
# STEP 6: Copy class mapping
# ============================================
if os.path.exists(CLASS_MAPPING):
    shutil.copy2(CLASS_MAPPING, f"{LOCAL_DATA_DIR}/class_mapping.json")
elif os.path.exists(f"{SYNTHETIC_DIR}/class_mapping.json"):
    shutil.copy2(f"{SYNTHETIC_DIR}/class_mapping.json", f"{LOCAL_DATA_DIR}/class_mapping.json")
elif os.path.exists(f"{REAL_IMAGES_DIR}/class_mapping.json"):
    shutil.copy2(f"{REAL_IMAGES_DIR}/class_mapping.json", f"{LOCAL_DATA_DIR}/class_mapping.json")

# ============================================
# SUMMARY
# ============================================
total_train = len(os.listdir(f"{LOCAL_DATA_DIR}/images/train"))
total_val = len(os.listdir(f"{LOCAL_DATA_DIR}/images/val"))

print(f"\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Training images: {total_train}")
print(f"  - Real: {real_count}")
print(f"  - Synthetic: {synthetic_to_use}")
print(f"Validation images: {total_val}")
print(f"\nReal image ratio: {real_count/total_train*100:.1f}%")
print("="*60)

## Cell 5: Convert to COCO Format

In [ ]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

def yolo_to_coco(images_dir, labels_dir, class_mapping_path, output_path):
    """
    Convert YOLO annotations to COCO format.
    Includes all required COCO fields to prevent pycocotools errors.
    """
    with open(class_mapping_path, 'r') as f:
        class_mapping = json.load(f)

    categories = [
        {
            "id": c["id"],
            "name": c["name"],
            "supercategory": c.get("category", "hazmat")
        }
        for c in class_mapping
    ]

    images = []
    annotations = []
    annotation_id = 0
    negative_count = 0

    image_files = list(Path(images_dir).glob("*.jpg")) + list(Path(images_dir).glob("*.png"))

    for img_id, img_path in enumerate(tqdm(sorted(image_files), desc="Converting")):
        with Image.open(img_path) as img:
            width, height = img.size

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": width,
            "height": height,
            "license": 1,
            "date_captured": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if not label_path.exists():
            negative_count += 1
            continue

        with open(label_path, 'r') as f:
            content = f.read().strip()

        if not content:
            negative_count += 1
            continue

        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            box_width = float(parts[3])
            box_height = float(parts[4])

            x = (x_center - box_width / 2) * width
            y = (y_center - box_height / 2) * height
            w = box_width * width
            h = box_height * height

            annotations.append({
                "id": annotation_id,
                "image_id": img_id,
                "category_id": class_id,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            annotation_id += 1

    coco_format = {
        "info": {
            "description": "Hazmat Label Detection - Confidence Boost Dataset",
            "url": "https://github.com/hazmat-detection",
            "version": "3.0",
            "year": 2025,
            "contributor": "HazProML",
            "date_created": datetime.now().strftime("%Y-%m-%d")
        },
        "licenses": [
            {
                "id": 1,
                "name": "MIT License",
                "url": "https://opensource.org/licenses/MIT"
            }
        ],
        "categories": categories,
        "images": images,
        "annotations": annotations
    }

    with open(output_path, 'w') as f:
        json.dump(coco_format, f)

    return len(images), len(annotations), negative_count

# Convert training set
print("Converting training set...")
train_imgs, train_anns, train_neg = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/train',
    f'{LOCAL_DATA_DIR}/labels/train',
    f'{LOCAL_DATA_DIR}/class_mapping.json',
    f'{LOCAL_DATA_DIR}/train.json'
)
print(f"  Images: {train_imgs} ({train_neg} negative samples)")
print(f"  Annotations: {train_anns}")

# Convert validation set
print("\nConverting validation set...")
val_imgs, val_anns, val_neg = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/val',
    f'{LOCAL_DATA_DIR}/labels/val',
    f'{LOCAL_DATA_DIR}/class_mapping.json',
    f'{LOCAL_DATA_DIR}/val.json'
)
print(f"  Images: {val_imgs} ({val_neg} negative samples)")
print(f"  Annotations: {val_anns}")

## Cell 6: Create YOLOX Experiment Config

### Key Modifications for Confidence Boost:

1. **`obj_loss_weight`** - Increased from 1.0 to 2.0 (or higher)
2. **`basic_lr_per_img`** - Reduced from 0.01/64 to 0.0005/64 (20x lower)
3. **`no_aug_epochs`** - Extended from 15 to 40 epochs
4. **`warmup_epochs`** - Reduced from 5 to 2 (preserving learned features)

In [ ]:
# Model configuration for YOLOX-Tiny
MODEL_CONFIGS = {
    "nano": {"depth": 0.33, "width": 0.25},
    "tiny": {"depth": 0.33, "width": 0.375},
    "s": {"depth": 0.33, "width": 0.50},
    "m": {"depth": 0.67, "width": 0.75},
}

MODEL_SIZE = "tiny"
config = MODEL_CONFIGS[MODEL_SIZE]

# Create experiment config with CONFIDENCE BOOST modifications
exp_content = f'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YOLOX Experiment Config - CONFIDENCE BOOST Fine-Tuning

Key modifications:
1. Increased obj_loss weight (2x) - Stronger objectness learning
2. Lower learning rate (20x) - Preserve learned features
3. Extended no_aug phase (40 epochs) - Better confidence calibration
4. Shorter warmup (2 epochs) - Quick adaptation from checkpoint
"""

import os
import torch
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()

        # ---------------- Model Architecture ----------------
        self.depth = {config["depth"]}
        self.width = {config["width"]}
        self.num_classes = {NUM_CLASSES}
        self.act = "silu"

        # ---------------- Dataset ----------------
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 4

        # ---------------- Training Schedule ----------------
        self.max_epoch = {MAX_EPOCHS}

        # CONFIDENCE BOOST: Shorter warmup (preserve learned features)
        self.warmup_epochs = 2

        # CONFIDENCE BOOST: 20x lower learning rate
        self.basic_lr_per_img = {BASIC_LR}

        self.scheduler = "yoloxwarmcos"
        self.min_lr_ratio = 0.01  # Even lower final LR
        self.weight_decay = 5e-4
        self.momentum = 0.9

        # CONFIDENCE BOOST: Extended no-aug phase (40 epochs)
        # This helps the model learn precise confidence scores
        self.no_aug_epochs = {NO_AUG_EPOCHS}

        # ---------------- Loss Weights ----------------
        # CONFIDENCE BOOST: Increase objectness loss weight
        # Default is 1.0, we use {OBJ_LOSS_WEIGHT} to force stronger objectness
        # This makes the model produce higher confidence for true positives

        # Note: These are accessed in yolox/models/yolo_head.py
        # We'll set them after model creation

        # ---------------- Input Size ----------------
        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.random_size = (14, 26)  # Multi-scale training

        # ---------------- Augmentation ----------------
        # Lighter augmentation for fine-tuning
        self.mosaic_prob = 0.5
        self.mixup_prob = 0.3
        self.enable_mixup = True
        self.mosaic_scale = (0.5, 1.5)
        self.mixup_scale = (0.5, 1.5)
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 5.0
        self.translate = 0.05
        self.shear = 1.0

        # ---------------- NMS & Confidence ----------------
        self.nmsthre = 0.65
        self.test_conf = 0.01

        # ---------------- Output & Evaluation ----------------
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_confidence_boost"

        # Disable mid-training evaluation to prevent crashes
        self.eval_interval = 1000

        self.save_history_ckpt = True
        self.print_interval = 50

    def get_model(self):
        """
        Override to set custom loss weights after model creation.
        """
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        in_channels = [256, 512, 1024]
        backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act)
        head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act)

        # CONFIDENCE BOOST: Set custom loss weights
        # Increase objectness loss weight to force stronger confidence scores
        if hasattr(head, 'use_l1'):
            pass  # L1 loss is handled automatically

        self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)

        return self.model
'''

# Write config
config_path = '/content/YOLOX/exps/confidence_boost_exp.py'
with open(config_path, 'w') as f:
    f.write(exp_content)

print(f"Created YOLOX config: {config_path}")
print(f"\n" + "="*60)
print("CONFIDENCE BOOST CONFIG SUMMARY")
print("="*60)
print(f"Model: YOLOX-{MODEL_SIZE}")
print(f"  Depth: {config['depth']}")
print(f"  Width: {config['width']}")
print(f"  Classes: {NUM_CLASSES}")
print(f"\nTraining:")
print(f"  Total epochs: {MAX_EPOCHS}")
print(f"  No-aug epochs: {NO_AUG_EPOCHS} (starts at epoch {MAX_EPOCHS - NO_AUG_EPOCHS})")
print(f"  Learning rate: {BASIC_LR * 64:.6f} (base)")
print(f"  Warmup: 2 epochs")
print(f"\nLoss weights:")
print(f"  Objectness: {OBJ_LOSS_WEIGHT}x (will be applied via trainer modification)")
print("="*60)

## Cell 7: Modify YOLOX Loss Weights

YOLOX's loss weights are hardcoded in `yolox/models/yolo_head.py`. We need to patch this file to increase the objectness loss weight.

**The key line is:**
```python
loss_obj = (self.bcewithlog_loss(obj_preds.view(-1, 1), obj_targets)).sum() / num_fg
```

We'll multiply this by `OBJ_LOSS_WEIGHT`.

In [ ]:
import re

yolo_head_path = '/content/YOLOX/yolox/models/yolo_head.py'

# Read the file
with open(yolo_head_path, 'r') as f:
    content = f.read()

# Check if already modified
if f'OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}' in content:
    print(f"yolo_head.py already modified with OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}")
else:
    # Find the loss computation section and add weight multiplier
    # Original pattern:
    #   loss_obj = (self.bcewithlog_loss(obj_preds.view(-1, 1), obj_targets)).sum() / num_fg

    # We'll add a constant at the top of the file and multiply loss_obj

    # Add constant after imports
    import_section_end = content.find('class YOLOXHead')
    if import_section_end > 0:
        new_content = content[:import_section_end]
        new_content += f'''
# CONFIDENCE BOOST: Increased objectness loss weight
# This forces the model to produce stronger objectness scores
OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}

'''
        new_content += content[import_section_end:]
        content = new_content

    # Modify the loss_obj line to multiply by OBJ_LOSS_WEIGHT
    # Pattern 1: Standard pattern
    old_pattern = r'(loss_obj\s*=\s*\(self\.bcewithlog_loss\(obj_preds\.view\(-1,\s*1\),\s*obj_targets\)\)\.sum\(\)\s*/\s*num_fg)'
    new_pattern = r'\1 * OBJ_LOSS_WEIGHT  # CONFIDENCE BOOST'

    if re.search(old_pattern, content):
        content = re.sub(old_pattern, new_pattern, content)
        print(f"Modified loss_obj line with OBJ_LOSS_WEIGHT = {OBJ_LOSS_WEIGHT}")
    else:
        # Try alternate pattern (some YOLOX versions have different formatting)
        # Look for any line that computes loss_obj and add multiplier at the end
        lines = content.split('\n')
        modified = False
        for i, line in enumerate(lines):
            if 'loss_obj' in line and 'bcewithlog_loss' in line and 'OBJ_LOSS_WEIGHT' not in line:
                # Add multiplier before any existing comment or at end
                if '#' in line:
                    comment_pos = line.index('#')
                    lines[i] = line[:comment_pos].rstrip() + ' * OBJ_LOSS_WEIGHT  # CONFIDENCE BOOST ' + line[comment_pos:]
                else:
                    lines[i] = line.rstrip() + ' * OBJ_LOSS_WEIGHT  # CONFIDENCE BOOST'
                modified = True
                print(f"Modified line {i+1}: {lines[i][:80]}...")
                break

        if modified:
            content = '\n'.join(lines)
        else:
            print("WARNING: Could not find loss_obj line to modify!")
            print("You may need to manually edit yolox/models/yolo_head.py")
            print("Find the line with 'loss_obj' and multiply by 2.0")

    # Write modified content
    with open(yolo_head_path, 'w') as f:
        f.write(content)

    print(f"\nSaved modified yolo_head.py")

# Verify modification
with open(yolo_head_path, 'r') as f:
    content = f.read()

if 'OBJ_LOSS_WEIGHT' in content:
    print(f"\nVerification: OBJ_LOSS_WEIGHT found in yolo_head.py")
    # Show the relevant lines
    for i, line in enumerate(content.split('\n')):
        if 'OBJ_LOSS_WEIGHT' in line:
            print(f"  Line {i+1}: {line[:100]}")

## Cell 8: Start Confidence Boost Fine-Tuning

This will:
1. Resume from your 100-epoch checkpoint
2. Train for 60 more epochs with:
   - 2x objectness loss weight
   - 20x lower learning rate
   - 40 epochs of no-augmentation phase
   - 40% real images in training data

**Expected time:** 2-4 hours on T4 GPU

In [ ]:
import threading
import time
import shutil
import os
import json

# Ensure COCO JSON files have required fields
for split in ['train', 'val']:
    json_path = f'{LOCAL_DATA_DIR}/{split}.json'
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)

        modified = False
        if 'info' not in data:
            data['info'] = {"description": "Hazmat Confidence Boost Dataset", "version": "3.0", "year": 2025}
            modified = True
        if 'licenses' not in data:
            data['licenses'] = [{"id": 1, "name": "MIT", "url": ""}]
            modified = True

        if modified:
            with open(json_path, 'w') as f:
                json.dump(data, f)
            print(f"Fixed {split}.json")

# Setup output directory
YOLOX_OUTPUT_DIR = '/content/outputs/yolox_confidence_boost'
os.makedirs(YOLOX_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy pre-trained checkpoint to resume from
print(f"Setting up checkpoint for resume...")
resume_ckpt = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"
shutil.copy(ckpt_path, resume_ckpt)
print(f"  Copied: {ckpt_path}")
print(f"  To: {resume_ckpt}")

# Check checkpoint epoch
ckpt_data = torch.load(resume_ckpt, map_location='cpu')
start_epoch = ckpt_data.get('start_epoch', 0)
print(f"  Starting from epoch: {start_epoch}")

In [ ]:
# Reset epoch counter so we train for full MAX_EPOCHS
# (Otherwise it would skip to epoch 100+ and do nothing)

print("Resetting checkpoint epoch counter for fine-tuning...")
ckpt_data = torch.load(resume_ckpt, map_location='cpu')
original_epoch = ckpt_data.get('start_epoch', 0)
ckpt_data['start_epoch'] = 0
torch.save(ckpt_data, resume_ckpt)
print(f"  Reset epoch from {original_epoch} to 0")
print(f"  Model weights preserved, will train for {MAX_EPOCHS} new epochs")

In [ ]:
# Checkpoint backup thread
backup_running = True

def backup_checkpoints():
    """Background thread to backup checkpoints to Google Drive."""
    output_dir = YOLOX_OUTPUT_DIR
    last_backup_epoch = -1

    while backup_running:
        time.sleep(60)

        if not os.path.exists(output_dir):
            continue

        latest_ckpt = os.path.join(output_dir, 'latest_ckpt.pth')
        if os.path.exists(latest_ckpt):
            try:
                ckpt = torch.load(latest_ckpt, map_location='cpu')
                current_epoch = ckpt.get('start_epoch', 0)

                if current_epoch > 0 and current_epoch % BACKUP_INTERVAL == 0 and current_epoch != last_backup_epoch:
                    backup_path = os.path.join(DRIVE_OUTPUT, f'confidence_boost_epoch_{current_epoch}.pth')
                    shutil.copy(latest_ckpt, backup_path)
                    print(f"\n Backup saved: confidence_boost_epoch_{current_epoch}.pth")
                    last_backup_epoch = current_epoch

                    shutil.copy(latest_ckpt, os.path.join(DRIVE_OUTPUT, 'confidence_boost_latest.pth'))
            except Exception as e:
                pass

# Start backup thread
backup_thread = threading.Thread(target=backup_checkpoints, daemon=True)
backup_thread.start()
print(f"Checkpoint backup thread started")

# Print training summary
print("\n" + "="*70)
print("CONFIDENCE BOOST FINE-TUNING")
print("="*70)
print(f"\nGoal: Fix confidence ceiling (~53% max) while preserving accuracy")
print(f"\nModifications applied:")
print(f"  1. Objectness loss weight: {OBJ_LOSS_WEIGHT}x (forces stronger confidence)")
print(f"  2. Learning rate: {BASIC_LR * 64:.6f} (20x lower, preserves features)")
print(f"  3. No-aug phase: {NO_AUG_EPOCHS} epochs (better calibration)")
print(f"  4. Real image ratio: {REAL_IMAGE_RATIO*100:.0f}% (realistic patterns)")
print(f"\nTraining: {MAX_EPOCHS} epochs on {train_imgs} images")
print(f"Estimated time: 2-4 hours on T4 GPU")
print(f"\nCheckpoints saved to: {DRIVE_OUTPUT}")
print("-"*70 + "\n")

# Start training
%cd /content/YOLOX

!PYTHONPATH=/content/YOLOX python tools/train.py \
    -f exps/confidence_boost_exp.py \
    -d 1 \
    -b {BATCH_SIZE} \
    --fp16 \
    -o \
    --resume

# Stop backup thread
backup_running = False

print("\n" + "="*70)
print("CONFIDENCE BOOST TRAINING COMPLETE!")
print("="*70)

## Cell 9: Save Final Checkpoints

In [ ]:
import shutil
from datetime import datetime

if os.path.exists(YOLOX_OUTPUT_DIR):
    print("Saving final checkpoints to Google Drive...")

    timestamp = datetime.now().strftime("%Y%m%d")

    files_to_save = [
        ('latest_ckpt.pth', f'confidence_boost_{MAX_EPOCHS}epoch_{timestamp}.pth'),
        ('latest_ckpt.pth', 'confidence_boost_final.pth'),
    ]

    for src_name, dst_name in files_to_save:
        src = os.path.join(YOLOX_OUTPUT_DIR, src_name)
        if os.path.exists(src):
            dst = os.path.join(DRIVE_OUTPUT, dst_name)
            shutil.copy(src, dst)
            size_mb = os.path.getsize(src) / 1024 / 1024
            print(f"  {dst_name} ({size_mb:.1f} MB)")

    print(f"\nAll checkpoints saved to: {DRIVE_OUTPUT}")
    print("\nFiles:")
    for f in sorted(os.listdir(DRIVE_OUTPUT)):
        fpath = os.path.join(DRIVE_OUTPUT, f)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("No training output found. Run training first.")

## Cell 10: Test Confidence Improvement

Run inference and compare confidence scores to verify the improvement.

In [ ]:
import json
import torch
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# Load class names
with open(f'{LOCAL_DATA_DIR}/class_mapping.json', 'r') as f:
    class_mapping = json.load(f)
HAZMAT_CLASSES = [c['name'] for c in class_mapping]

# Find the new checkpoint
new_ckpt_path = os.path.join(DRIVE_OUTPUT, 'confidence_boost_final.pth')
if not os.path.exists(new_ckpt_path):
    new_ckpt_path = os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth')

print(f"Testing checkpoint: {new_ckpt_path}")

# Load model
from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

exp = get_exp('/content/YOLOX/exps/confidence_boost_exp.py', None)
model = exp.get_model()
ckpt = torch.load(new_ckpt_path, map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda()
model.eval()

print(f"Model loaded (epoch {ckpt.get('start_epoch', 'unknown')})")

# Inference settings
conf_thresh = 0.25
nms_thresh = 0.45
input_size = (640, 640)
preproc = ValTransform(legacy=False)

# Run on validation images
val_images = list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.jpg'))[:20]
if len(val_images) < 20:
    val_images += list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.png'))[:20-len(val_images)]

all_confidences = []
detection_counts = []

print(f"\nRunning inference on {len(val_images)} images...")

for img_path in tqdm(val_images, desc="Inference"):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    img_tensor, _ = preproc(img, None, input_size)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).float().cuda()

    with torch.no_grad():
        outputs = model(img_tensor)
        outputs = postprocess(outputs, exp.num_classes, conf_thresh, nms_thresh)

    if outputs[0] is not None:
        output = outputs[0].cpu().numpy()
        scores = output[:, 4] * output[:, 5]  # objectness * class_conf
        all_confidences.extend(scores.tolist())
        detection_counts.append(len(scores))
    else:
        detection_counts.append(0)

In [ ]:
# Analyze confidence distribution
print("\n" + "="*60)
print("CONFIDENCE BOOST RESULTS")
print("="*60)

if len(all_confidences) > 0:
    all_confidences = np.array(all_confidences)

    print(f"\nConfidence Statistics:")
    print(f"  Total detections: {len(all_confidences)}")
    print(f"  Min confidence: {all_confidences.min()*100:.1f}%")
    print(f"  Max confidence: {all_confidences.max()*100:.1f}%")
    print(f"  Mean confidence: {all_confidences.mean()*100:.1f}%")
    print(f"  Median confidence: {np.median(all_confidences)*100:.1f}%")

    # Check if we broke the 53% ceiling
    above_53 = (all_confidences > 0.53).sum()
    above_70 = (all_confidences > 0.70).sum()
    above_90 = (all_confidences > 0.90).sum()

    print(f"\nConfidence Distribution:")
    print(f"  Above 53%: {above_53} ({above_53/len(all_confidences)*100:.1f}%)")
    print(f"  Above 70%: {above_70} ({above_70/len(all_confidences)*100:.1f}%)")
    print(f"  Above 90%: {above_90} ({above_90/len(all_confidences)*100:.1f}%)")

    # Success criteria
    if all_confidences.max() > 0.70:
        print(f"\n SUCCESS: Confidence ceiling broken!")
        print(f"  Previous max: ~53%")
        print(f"  New max: {all_confidences.max()*100:.1f}%")
    else:
        print(f"\n  Confidence still capped at {all_confidences.max()*100:.1f}%")
        print(f"  Consider: Higher obj_loss weight (3.0) or more training")

    # Plot histogram
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(all_confidences * 100, bins=20, edgecolor='black', alpha=0.7)
    plt.axvline(x=53, color='r', linestyle='--', label='Previous ceiling (53%)')
    plt.xlabel('Confidence (%)')
    plt.ylabel('Count')
    plt.title('Confidence Distribution')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.bar(range(len(detection_counts)), detection_counts, alpha=0.7)
    plt.xlabel('Image Index')
    plt.ylabel('Detections')
    plt.title('Detections per Image')

    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_OUTPUT, 'confidence_analysis.png'), dpi=150)
    plt.show()

    print(f"\nAnalysis saved to: {DRIVE_OUTPUT}/confidence_analysis.png")
else:
    print("No detections found. Check model loading and thresholds.")

print("="*60)

## Cell 11: Visual Comparison

Show sample detections with confidence scores.

In [ ]:
# Visualization colors
np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(NUM_CLASSES, 3), dtype=np.uint8)

def visualize_detections(img, boxes, scores, cls_ids):
    """Draw bounding boxes with confidence scores."""
    img_vis = img.copy()
    for i in range(len(boxes)):
        box = boxes[i]
        cls_id = int(cls_ids[i]) if cls_ids[i] < len(HAZMAT_CLASSES) else 0
        score = scores[i]
        x0, y0, x1, y1 = map(int, box)

        # Color based on confidence
        if score > 0.7:
            color = (0, 255, 0)  # Green for high confidence
        elif score > 0.5:
            color = (255, 255, 0)  # Yellow for medium
        else:
            color = (0, 165, 255)  # Orange for low

        cv2.rectangle(img_vis, (x0, y0), (x1, y1), color, 2)
        label = f'{HAZMAT_CLASSES[cls_id][:15]}: {score*100:.0f}%'
        txt_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
        cv2.rectangle(img_vis, (x0, y0 - txt_size[1] - 4), (x0 + txt_size[0], y0), color, -1)
        cv2.putText(img_vis, label, (x0, y0 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return img_vis

# Run on sample images and display
sample_images = val_images[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_images):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    img_tensor, _ = preproc(img, None, input_size)
    img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).float().cuda()

    with torch.no_grad():
        outputs = model(img_tensor)
        outputs = postprocess(outputs, exp.num_classes, 0.25, 0.45)

    if outputs[0] is not None:
        output = outputs[0].cpu().numpy()
        boxes = output[:, :4]
        ratio = min(input_size[0] / h, input_size[1] / w)
        boxes /= ratio
        scores = output[:, 4] * output[:, 5]
        cls_ids = output[:, 6]
        img = visualize_detections(img, boxes, scores, cls_ids)
        title = f'{len(boxes)} det, max: {scores.max()*100:.0f}%'
    else:
        title = 'No detections'

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('Confidence Boost Model - Sample Detections', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'sample_detections.png'), dpi=150)
plt.show()

print(f"\nSample detections saved to: {DRIVE_OUTPUT}/sample_detections.png")

## Cell 12: Export Summary & Next Steps

In [ ]:
print("="*70)
print("CONFIDENCE BOOST FINE-TUNING COMPLETE")
print("="*70)

print(f"\nOutput directory: {DRIVE_OUTPUT}")
print("\nFiles created:")
if os.path.exists(DRIVE_OUTPUT):
    for f in sorted(os.listdir(DRIVE_OUTPUT)):
        fpath = os.path.join(DRIVE_OUTPUT, f)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"  {f} ({size_mb:.1f} MB)")

print(f"\n" + "-"*70)
print("MODIFICATIONS APPLIED:")
print("-"*70)
print(f"1. Objectness loss weight: {OBJ_LOSS_WEIGHT}x")
print(f"2. Learning rate: {BASIC_LR * 64:.6f} (20x lower)")
print(f"3. No-aug phase: {NO_AUG_EPOCHS} epochs")
print(f"4. Real image ratio: {REAL_IMAGE_RATIO*100:.0f}%")
print(f"5. Total training: {MAX_EPOCHS} epochs")

print(f"\n" + "-"*70)
print("NEXT STEPS:")
print("-"*70)
print("""
1. Export to ExecuTorch:
   - Run 04_export_executorch.ipynb
   - Update CHECKPOINT_PATH to: confidence_boost_final.pth

2. Test in HazProML app:
   - Copy .pte file to assets/models/
   - Rebuild: npx expo run:ios

3. If confidence still low:
   - Increase OBJ_LOSS_WEIGHT to 3.0
   - Add more real training images
   - Train for more epochs (100+)

4. If classification accuracy drops:
   - Lower learning rate further
   - Reduce OBJ_LOSS_WEIGHT to 1.5
""")
print("="*70)